# LAB DAY 19: GraphRAG — US Electric Vehicle Dataset

**Sinh viên:** Nguyen Hoang Minh - 2A202600963

## Dataset: 70 documents (`dataset/dataset/`) → 68 usable after cleaning

## Kết quả chính (Full LLM — cleaned corpus + graph-first pipeline)
- **Flat RAG:** 80.0% | **GraphRAG:** **100.0%** | **Multi-hop Graph:** **100.0%**
- Graph wins when Flat wrong: **4 questions** (Q6, Q12, Q13, Q17)
- Graph: 810 triples, 888 nodes, 761 edges
- Demo UI: `streamlit run streamlit_app.py`

## PHẦN 1: Nghiên cứu

### Entity Extraction: Phân biệt Node và Thuộc tính?
- **Node**: thực thể có thể kết nối nhiều quan hệ (OpenAI, Sam Altman)
- **Thuộc tính**: mô tả gắn với 1 node (không tạo node riêng nếu không cần multi-hop)
- LLM dùng schema quan hệ (FOUNDED_BY, CEO_OF) để chuẩn hóa

### Deduplication quan trọng vì?
- Tránh node trùng ("Google" vs "google")
- Giữ đồ thị gọn, BFS chính xác hơn

### BFS vs Vector Search?
- **Vector (Flat RAG)**: similarity ngữ nghĩa — mạnh khi fact nằm trong 1 chunk; dễ nhầm số gần nghĩa (58% vs 51%)
- **Graph (BFS + paths)**: keyword search + 4-hop BFS + shortest paths — mạnh multi-hop (BNEF → McKerracher, J.D. Power → 29.2% → Krear)

In [ ]:
# Setup
import os
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

PROJECT_ROOT = Path(".").resolve()
os.chdir(PROJECT_ROOT)

# Set API key: os.environ["OPENAI_API_KEY"] = "sk-..."
from dotenv import load_dotenv
load_dotenv()

print("API key set:", bool(os.getenv("OPENAI_API_KEY")))

## Bước 1: Entity & Relation Extraction (Indexing)

In [ ]:
from src.config import CORPUS_PATH, DATASET_DIR, TRIPLES_PATH
from src.corpus import load_dataset, prepare_corpus
from src.entity_extraction import extract_triples_from_corpus, load_triples, save_triples
from src.fact_triples import enrich_triples

docs = prepare_corpus(DATASET_DIR, CORPUS_PATH)
print(f"Loaded {len(docs)} usable documents (68/70 after PDF/binary filtering)")
print(docs[0].title[:80], "...")

demo = not os.getenv("OPENAI_API_KEY")
if TRIPLES_PATH.exists():
    triples = enrich_triples(load_triples(TRIPLES_PATH), CORPUS_PATH)
    print(f"Loaded + enriched {len(triples)} triples from {TRIPLES_PATH}")
else:
    result = extract_triples_from_corpus(demo=demo)
    triples = enrich_triples(result.triples, CORPUS_PATH)
    save_triples(triples, TRIPLES_PATH)
    print(f"Extracted {len(triples)} triples ({result.source})")

for t in triples[:5]:
    print(f"  {t}")

## Bước 2: Graph Construction (NetworkX)

In [ ]:
from src.graph_construction import build_networkx_graph, graph_stats, get_neighbors_bfs
from src.visualize import visualize_graph
from src.config import GRAPH_IMAGE_PATH

graph = build_networkx_graph(triples)
stats = graph_stats(graph)
print("Graph stats:", stats)

visualize_graph(graph, GRAPH_IMAGE_PATH)
display(Image(filename=str(GRAPH_IMAGE_PATH)))

## Bước 3: Querying (GraphRAG — multi-hop retrieval + FACT fallback)

In [ ]:
from src.querying import answer_with_graph
from src.flat_rag import FlatRAG
from src.config import DATASET_DIR

question = "Which company leads the US EV market with more than half of all EV sales?"
graph_ans = answer_with_graph(question, graph)
flat_rag = FlatRAG(dataset_dir=DATASET_DIR)
flat_rag.index()
flat_ans = flat_rag.answer(question)

print("Question:", question)
print("\n--- GraphRAG ---")
print(graph_ans.answer)
print("\n--- Flat RAG ---")
print(flat_ans.answer)

## Bước 4: Evaluation — 20 câu hỏi benchmark (US EV dataset)

| Metric | Flat RAG | GraphRAG |
|--------|----------|----------|
| Overall accuracy | **80.0%** | **100.0%** |
| Multi-hop accuracy | **85.7%** | **100.0%** |
| Graph wins (Flat wrong) | — | **4** (Q6, Q12, Q13, Q17) |
| Both correct | 16 | 16 |

**Kết luận:** Sau khi clean corpus + graph-first pipeline (fact enrichment, multi-hop paths, direct triple fallback), GraphRAG vượt Flat RAG — đặc biệt khi Flat nhầm số liệu gần nghĩa trong chunks.

In [ ]:
from src.evaluation import run_evaluation, print_summary
from src.config import EVAL_RESULTS_PATH, COST_REPORT_PATH

df = run_evaluation(graph, flat_rag)
print_summary(df)
df

## Phân tích chi phí (Token & Time)

In [ ]:
from src.config import COST_REPORT_PATH, EVAL_RESULTS_PATH
from src.evaluation import compute_summary

# Đọc kết quả đã lưu (từ Streamlit hoặc python main.py)
if EVAL_RESULTS_PATH.exists():
    df = pd.read_csv(EVAL_RESULTS_PATH)
    summary = compute_summary(df)
    print("=== BENCHMARK SUMMARY ===")
    for k, v in summary.items():
        print(f"  {k}: {v}")

if COST_REPORT_PATH.exists():
    print("\n=== COST ANALYSIS ===")
    print(json.dumps(json.loads(COST_REPORT_PATH.read_text()), indent=2, ensure_ascii=False))

# GraphRAG đúng khi Flat RAG sai
graph_fixes = df[(df["flat_correct"] == "Sai") & (df["graph_correct"] == "Đúng")]
print(f"\nGraphRAG fixes Flat RAG errors: {len(graph_fixes)} cases")
if len(graph_fixes):
    display(graph_fixes[["id", "question", "flat_rag_answer", "graph_rag_answer"]])

# GraphRAG sai khi Flat RAG đúng
flat_wins = df[(df["flat_correct"] == "Đúng") & (df["graph_correct"] == "Sai")]
print(f"\nFlat RAG wins over GraphRAG: {len(flat_wins)} cases")
if len(flat_wins):
    display(flat_wins[["id", "question", "ground_truth", "graph_rag_answer"]])

## Streamlit Demo UI

```bash
streamlit run streamlit_app.py
```

Tabs: **Query** (so sánh Flat vs Graph) | **Graph** (visualize + BFS) | **Evaluation** (20 câu benchmark) | **Corpus**

## (Tùy chọn) Neo4j & NodeRAG

- **Neo4j**: `python main.py --neo4j` (cần Neo4j Desktop + password trong `.env`)
- **NodeRAG**: `python noderag_setup.py` rồi `pip install NodeRAG && python -m NodeRAG.build -f noderag_project`